<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Data Interoperability with Unity Catalog</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">5. Lakehouse Federation</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 5.5 Demo Implementing Lakehouse Federation

This demo walks through the **complete Lakehouse Federation workflow** against a Lakebase Postgres database: stand up a **Lakebase Autoscaling** project, seed a small operational **store-operations** table, federate it back into Unity Catalog as a foreign catalog, then run a federated query and a cross-system join with a UC-resident TPC-DS fact.

Lakebase stands in for an external Postgres instance - the federation mechanics (`CONNECTION` + `FOREIGN CATALOG`) are identical to what you would set up against any external Postgres.

## Learning Objectives

By the end of this demonstration, you will be able to:
- Stand up a Lakebase Autoscaling Postgres project and seed a table
- Create a Unity Catalog `CONNECTION` to a Postgres instance using credentials
- Register a `FOREIGN CATALOG` over a Postgres database
- Run a standalone federated query and a cross-system join with a UC-native fact table
- Inspect query pushdown to confirm what work was delegated to Postgres

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<div>
<strong style="color: #c62828">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333">
<li><strong>Serverless Compute, Version 5</strong>: <a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #2272B4">How to select an environment version</a></li>
</ul>
<p style="margin: 8px 0 0 0; color: #333"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. You must have run <strong>0 - Required Setup</strong> first. The Lakebase steps require <strong>Lakebase Autoscaling</strong> in your workspace region (see <a href="https://docs.databricks.com/aws/en/oltp/projects/manage-projects#availability" style="color: #2272B4">region availability</a>).</p>
</div>
</div>
</div>

<div style="font-size: 1em; border-left: 4px solid #7b1fa2; background: #f3e5f5; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #4a148c; font-size: 1.1em;">Lakebase Overview</strong>
            <ul style="margin: 8px 0 0 16px; color: #333">
                <li><strong>Lakebase</strong> is Databricks-managed PostgreSQL for OLTP workloads next to the lakehouse. <strong>Lakebase Autoscaling</strong> is the current generation, with autoscaling compute, scale-to-zero, branching, and instant restore.</li>
                <li>A <strong>project</strong> is the top-level container; each has one or more <strong>branches</strong> (<code>production</code> by default), and each branch holds a Postgres database (<code>databricks_postgres</code> by default).</li>
                <li>You connect with any standard Postgres client using a connection string; authentication is via Databricks OAuth (paste an OAuth token as the password). Lakebase also has its own SQL Editor in the workspace UI.</li>
            </ul>
            <p style="margin: 12px 0 0 0; color: #333;">The Lakebase provisioning and Postgres steps below are shown as guided walkthrough steps; the Unity Catalog steps run in this notebook once the connection details are filled in.</p>
        </div>
    </div>
</div>

In [0]:
%run ../Includes/Classroom-Setup-5

## A. Provision the Lakebase Database

Three steps in the Lakebase UI: create a project, capture the connection details, and create and seed the operational table.

### A1. Create a Lakebase Autoscaling Project

From the workspace **apps switcher** (the icon grid in the top-right), open the **Lakebase** app. Select **Autoscaling** and click **New project**. Accept the default Postgres version and create. The project is created with a single `production` branch and a default `databricks_postgres` database.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Lakebase: Apps Switcher Screenshot (click to expand)</span>
  </summary>
  <div style="padding: 16px; background: #fafafa;">
<img src="https://files.training.databricks.com/binder/prod_main/data-interoperability-with-unity-catalog-en_us-2.1.0/images/20260915T185754Z/Data Interoperability with Unity Catalog/Includes/images/lakebase-apps-switcher.png" alt="Workspace apps switcher with the Lakebase tile highlighted" style="max-width: 100%; height: auto; border: 1px solid #d0d0d0; border-radius: 4px;"/>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Lakebase: Create Project Dialog Screenshot (click to expand)</span>
  </summary>
  <div style="padding: 16px; background: #fafafa;">
<img src="https://files.training.databricks.com/binder/prod_main/data-interoperability-with-unity-catalog-en_us-2.1.0/images/20260915T185754Z/Data Interoperability with Unity Catalog/Includes/images/lakebase-create-project-dialog.png" alt="Lakebase Autoscaling new project dialog" style="max-width: 100%; height: auto; border: 1px solid #d0d0d0; border-radius: 4px;"/>
  </div>
</details>

### A2. Capture the Connection Details

From your project, select the **production** branch and click **Connect**. Set the **Role** dropdown to your Databricks email (under OAuth roles). The dialog generates a `psql` connection string with the values you need for the UC `CONNECTION` in Section B.

| What you need | Where to find it | Example |
|---|---|---|
| **host** | The hostname in the connection string (between `@` and the next `/`) | `ep-abc-123.staging.cloud.databricks.com` |
| **port** | Always `5432` for Lakebase | `5432` |
| **database** | The database name (after the last `/`); used for the `FOREIGN CATALOG` | `databricks_postgres` |
| **user** | Your email address | `user@databricks.com` |
| **password** | Click **Copy OAuth token** at the bottom of the dialog | (a long token string) |

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Lakebase: Connect Dialog Screenshot (click to expand)</span>
  </summary>
  <div style="padding: 16px; background: #fafafa;">
<img src="https://files.training.databricks.com/binder/prod_main/data-interoperability-with-unity-catalog-en_us-2.1.0/images/20260915T185754Z/Data Interoperability with Unity Catalog/Includes/images/lakebase-connect-oauth.png" alt="Lakebase Connect dialog showing the OAuth psql connection string" style="max-width: 100%; height: auto; border: 1px solid #d0d0d0; border-radius: 4px;"/>
  </div>
</details>

### A3. Create and Seed the `store_ops` Table

Open the Lakebase **SQL Editor** for your `production` branch and run the SQL below. It creates a small `store_ops` table keyed on `s_store_sk` (the same surrogate key TPC-DS uses for stores) and seeds 20 rows that overlap the TPC-DS store space.

In a real OLTP system this table would hold operational attributes - store manager, remodel status, staffing level, region - that change frequently and live in the transactional store. Joining it with the analytical fact table on the lakehouse side is the canonical Lakehouse Federation use case.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Lakebase: Create + seed store_ops (click to expand)</span>
  </summary>
  <div style="padding: 16px; background: #fafafa;">
<div class="code-block" data-language="sql">
-- Run in the Lakebase SQL Editor against the production branch / databricks_postgres database.
CREATE SCHEMA IF NOT EXISTS tpcds_ops;

-- Drop and recreate so this block can be re-run safely.
DROP TABLE IF EXISTS tpcds_ops.store_ops;

CREATE TABLE tpcds_ops.store_ops (
    s_store_sk      BIGINT       PRIMARY KEY,
    store_manager   VARCHAR(64)  NOT NULL,
    remodel_status  VARCHAR(16)  NOT NULL,
    staffing_level  VARCHAR(16)  NOT NULL,
    region          VARCHAR(16)  NOT NULL,
    updated_at      TIMESTAMPTZ  NOT NULL DEFAULT now()
);

-- Seed 20 rows. s_store_sk 1..20 line up with samples.tpcds_sf1000.store,
-- so the cross-system join in Section D has matches on both sides.
INSERT INTO tpcds_ops.store_ops
    (s_store_sk, store_manager, remodel_status, staffing_level, region)
SELECT
    sk,
    'Manager ' || sk,
    CASE (sk % 3) WHEN 0 THEN 'planned' WHEN 1 THEN 'in_progress' ELSE 'complete' END,
    CASE (sk % 3) WHEN 0 THEN 'lean'    WHEN 1 THEN 'standard'    ELSE 'full' END,
    CASE (sk % 4) WHEN 0 THEN 'west' WHEN 1 THEN 'south' WHEN 2 THEN 'midwest' ELSE 'northeast' END
FROM generate_series(1, 20) AS s(sk);

-- Smoke check
SELECT region, COUNT(*) AS stores
FROM tpcds_ops.store_ops
GROUP BY region
ORDER BY region;
</div>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Lakebase: SQL Editor Screenshot (click to expand)</span>
  </summary>
  <div style="padding: 16px; background: #fafafa;">
<img src="https://files.training.databricks.com/binder/prod_main/data-interoperability-with-unity-catalog-en_us-2.1.0/images/20260915T185754Z/Data Interoperability with Unity Catalog/Includes/images/lakebase-sql-create-table.png" alt="Lakebase SQL Editor with CREATE TABLE / INSERT statements" style="max-width: 100%; height: auto; border: 1px solid #d0d0d0; border-radius: 4px;"/>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var label = lang === 'bash' ? 'Terminal' : 'Lakebase';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code; document.body.appendChild(t); t.select();
            document.execCommand('copy'); document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

### A4. Materialize the UC-Side Fact Slice

The cross-system join needs a governed UC table on the analytical side. This cell (which **runs here in the notebook**) builds a small, pre-aggregated slice of TPC-DS `store_sales` - net revenue per store for a 30-day window - into your course schema. Keeping it pre-aggregated keeps the demo fast.

In [0]:
CREATE OR REPLACE TABLE data_interoperability_tpcds.demo_store_sales_by_store AS
SELECT
  ss_store_sk,
  COUNT(*)                    AS line_items,
  ROUND(SUM(ss_net_paid), 2)  AS net_revenue
FROM samples.tpcds_sf1000.store_sales
WHERE ss_sold_date_sk BETWEEN 2451180 AND 2451210
  AND ss_store_sk IS NOT NULL
GROUP BY ss_store_sk;

## B. Federate Lakebase from Unity Catalog

With the Lakebase table in place, the federation side is the standard two-object pattern: a `CONNECTION` (with credentials) and a `FOREIGN CATALOG` (the UC namespace surface). Fill in the host, user, and OAuth token you captured in A2, then run the two cells.


<div style="border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <span style="font-size: 24px;"></span>
        <div>
            <strong style="color: #e65100; font-size: 1.1em;">Update the following query before executing </strong>
            <p style="margin: 8px 0 0 0; color: #333;">Replace the placeholder values in the following query with those captured from the Lakebase Connect dialog (Section A2).</p>
        </div>
    </div>
</div>

In [0]:
-- Fill in the values captured from the Lakebase Connect dialog (Section A2).
-- For production, prefer secret('<scope>','<key>') over plaintext credentials.
-- In a shared workspace, add a unique suffix to the connection name to avoid collisions.
CREATE CONNECTION IF NOT EXISTS demo_store_ops_pg
  TYPE postgresql
  OPTIONS (
    host '<lakebase-host>',
    port '5432',
    user '<your-databricks-email>',
    password '<oauth-token>'
  );

In [0]:
DESCRIBE CONNECTION EXTENDED demo_store_ops_pg;

In [0]:
CREATE FOREIGN CATALOG IF NOT EXISTS demo_store_ops_catalog
  USING CONNECTION demo_store_ops_pg
  OPTIONS (database 'databricks_postgres');

In [0]:
SHOW SCHEMAS IN demo_store_ops_catalog;

## C. Federated Query

A standalone aggregate against the foreign catalog. This reads exclusively from Lakebase - the aggregation is pushed down to Postgres.

In [0]:
SELECT region, COUNT(*) AS stores
FROM demo_store_ops_catalog.tpcds_ops.store_ops
GROUP BY region
ORDER BY stores DESC;

## D. Cross-System Join - Lakebase + Unity Catalog

The headline use case. Federated `store_ops` from Lakebase joined with the UC-resident `demo_store_sales_by_store` slice, aggregated to net revenue by region. Unity Catalog governs both halves of the join: the same grants, audit, and lineage apply across systems.

In [0]:
SELECT
  so.region,
  COUNT(*)                        AS stores,
  SUM(ss.line_items)              AS line_items,
  ROUND(SUM(ss.net_revenue), 2)   AS net_revenue
FROM data_interoperability_tpcds.demo_store_sales_by_store ss
JOIN demo_store_ops_catalog.tpcds_ops.store_ops so
  ON so.s_store_sk = ss.ss_store_sk
GROUP BY so.region
ORDER BY net_revenue DESC;

## E. Inspect Pushdown

`EXPLAIN FORMATTED` reveals which operations the planner pushed down to Postgres. For Postgres sources, filters, projections, limits, and aggregates push down on Databricks Runtime 13.3+ / SQL warehouses 2023.40+.

In [0]:
EXPLAIN FORMATTED
SELECT region, COUNT(*)
FROM demo_store_ops_catalog.tpcds_ops.store_ops
WHERE region IN ('west', 'south')
GROUP BY region;

## F. Cleanup

Remove the UC objects created by this demo. Also delete the Lakebase `store_ops` table (and the project, if you created it only for this demo) from the Lakebase UI.

In [0]:
DROP CATALOG IF EXISTS demo_store_ops_catalog;
DROP CONNECTION IF EXISTS demo_store_ops_pg;
DROP TABLE IF EXISTS data_interoperability_tpcds.demo_store_sales_by_store;

## Demo Complete

This demo covered the full Lakehouse Federation workflow against Lakebase Postgres: provisioning the database, creating a UC `CONNECTION` and `FOREIGN CATALOG`, running a federated query and a cross-system join, and confirming pushdown with `EXPLAIN FORMATTED`.

<div style="font-size: 1em; border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #2e7d32; font-size: 1.1em;">Your Turn: the 5.6 Lab</strong>
            <p style="margin: 8px 0 0 0; color: #333;">You just saw the complete workflow on the <strong>store-operations</strong> dataset. In the <strong>5.6 Lab</strong> you will implement the same pattern yourself against a <strong>customer-enrichment</strong> dataset - provisioning Lakebase, creating the <code>CONNECTION</code> and <code>FOREIGN CATALOG</code>, and running a cross-system join with the TPC-DS store sales fact.</p>
        </div>
    </div>
</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>